# 🎓 Análisis académico y alerta temprana de deserción
### Proyecto Integrador · Base de Datos III

Este notebook consume el **Data Warehouse (capa Gold)** en PostgreSQL y presenta el análisis
descriptivo, comparativo, de patrones y el modelo predictivo de riesgo de abandono.

| Etapa | Tecnología | Resultado |
|---|---|---|
| 🥉 Bronze | MongoDB Atlas | Datos crudos del CSV |
| 🥈 Silver | MongoDB Atlas | Datos limpios y tipados |
| 🥇 Gold | PostgreSQL | Modelo estrella + vistas analíticas |
| 📊 Análisis | Python / Plotly | KPIs, visualizaciones y modelo ML |

**Contenido**
1. Configuración y carga de datos
2. KPIs generales
3. Análisis descriptivo
4. Visualizaciones
5. Modelo predictivo de riesgo
6. Conclusiones
7. Exportación de gráficos


## 1 · Configuración y carga de datos

In [16]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import text
from IPython.display import display, HTML, Markdown

# --- Rutas y conexión al Data Warehouse ---
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE_DIR))
from src.conn import get_pg_engine

pg_engine = get_pg_engine()
print(f"✅ Data Warehouse: {pg_engine.url.database}")

# --- Paleta y estilo visual ---
PALETA_CARRERAS = ["#2563EB", "#F59E0B", "#10B981", "#8B5CF6", "#EC4899", "#14B8A6"]
COLOR_CARRERA = {}  # se completa al leer las carreras del DW (soporta N carreras)
COLOR_RIESGO = {
    "Riesgo Alto": "#DC2626",
    "Riesgo Medio": "#F59E0B",
    "Riesgo Bajo": "#16A34A",
}
TEXTO = "#0F172A"
GRIS = "#64748B"


def estilo_fig(fig, titulo, subtitulo="", height=480, grid="y"):
    """Aplica el estilo visual común a todas las figuras."""
    fig.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>{titulo}</b>"
            + (f"<br><sup style='color:{GRIS}'>{subtitulo}</sup>" if subtitulo else ""),
            x=0.01,
            xanchor="left",
            font=dict(size=20, color=TEXTO),
        ),
        font=dict(family="Segoe UI, Arial, sans-serif", size=13, color="#334155"),
        paper_bgcolor="white",
        plot_bgcolor="white",
        height=height,
        margin=dict(l=20, r=30, t=95 if subtitulo else 70, b=45),
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
            title=dict(text=""), traceorder="normal",
        ),
        hoverlabel=dict(bgcolor="white", font_size=12, font_family="Segoe UI, Arial"),
    )
    fig.update_xaxes(
        showgrid=(grid in ("x", "both")), gridcolor="#F1F5F9", linecolor="#E2E8F0"
    )
    fig.update_yaxes(
        showgrid=(grid in ("y", "both")), gridcolor="#F1F5F9", zeroline=False,
        linecolor="#E2E8F0",
    )
    return fig


print("✅ Configuración lista.")


✅ Data Warehouse: dw_universidad
✅ Configuración lista.


In [17]:
# Cargamos las vistas analíticas del Data Warehouse (usar text() evita el
# problema con los caracteres % en SQL cuando se usa psycopg2).
df_materias = pd.read_sql(
    text("SELECT * FROM dw_academico.vw_rendimiento_por_materia ORDER BY pct_no_aprobacion DESC"),
    pg_engine,
)
df_evolucion = pd.read_sql(
    text("SELECT * FROM dw_academico.vw_evolucion_periodo_carrera ORDER BY periodo_inscripcion"),
    pg_engine,
)
df_alertas = pd.read_sql(
    text("SELECT * FROM dw_academico.vw_alerta_temprana_alumnos ORDER BY score_riesgo DESC"),
    pg_engine,
)

print(
    f"✅ Datos cargados: {len(df_materias)} materias · "
    f"{len(df_evolucion)} registros de evolución · "
    f"{df_alertas['id_alumno'].nunique()} alumnos"
)

✅ Datos cargados: 6 materias · 28 registros de evolución · 200 alumnos


## 2 · KPIs generales

In [18]:
total_eval = int(df_materias["total_evaluaciones"].sum())
alumnos = int(df_alertas["id_alumno"].nunique())
promedio = (df_materias["promedio_nota"] * df_materias["total_evaluaciones"]).sum() / total_eval
no_aprobacion = (
    df_materias["pct_no_aprobacion"] * df_materias["total_evaluaciones"]
).sum() / total_eval
riesgo_alto = df_alertas["semaforo_riesgo"].eq("Riesgo Alto").mean() * 100


def tarjeta(titulo, valor, detalle, color):
    return f"""
    <div style="background:#fff;border:1px solid #E2E8F0;border-top:4px solid {color};
                border-radius:14px;padding:16px 18px;box-shadow:0 6px 16px rgba(15,23,42,.06);">
      <div style="font-size:11px;letter-spacing:.09em;text-transform:uppercase;
                  color:#64748B;font-weight:700;">{titulo}</div>
      <div style="font-size:30px;font-weight:800;color:{color};margin:6px 0 2px;">{valor}</div>
      <div style="font-size:12px;color:#94A3B8;">{detalle}</div>
    </div>"""


html = (
    '<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin:8px 0 18px;">'
    + tarjeta("Evaluaciones", f"{total_eval:,}", "registros en la tabla de hechos", "#2563EB")
    + tarjeta("Alumnos", f"{alumnos:,}", "estudiantes únicos analizados", "#7C3AED")
    + tarjeta("Promedio general", f"{promedio:.2f}", "escala 0 a 10", "#0891B2")
    + tarjeta("Riesgo alto", f"{riesgo_alto:.1f}%", "del total de alumnos", "#DC2626")
    + "</div>"
)
display(HTML(html))

## 3 · Análisis descriptivo

### 3.1 · Rendimiento por materia

In [19]:
tabla_materias = (
    df_materias[
        [
            "codigo_materia",
            "nombre_materia",
            "anio_cursada",
            "total_evaluaciones",
            "promedio_nota",
            "asistencia_media_pct",
            "pct_no_aprobacion",
            "pct_desaprobados_reales",
            "pct_ausentismo",
        ]
    ]
    .rename(
        columns={
            "codigo_materia": "Código",
            "nombre_materia": "Materia",
            "anio_cursada": "Año",
            "total_evaluaciones": "Evaluaciones",
            "promedio_nota": "Promedio",
            "asistencia_media_pct": "Asistencia %",
            "pct_no_aprobacion": "No aprobación %",
            "pct_desaprobados_reales": "Desaprobados %",
            "pct_ausentismo": "Ausentismo %",
        }
    )
    .style.format(
        {
            "Evaluaciones": "{:,.0f}",
            "Promedio": "{:.2f}",
            "Asistencia %": "{:.1f}",
            "No aprobación %": "{:.1f}",
            "Desaprobados %": "{:.1f}",
            "Ausentismo %": "{:.1f}",
        }
    )
    .background_gradient(subset=["No aprobación %"], cmap="Reds")
    .background_gradient(subset=["Promedio"], cmap="Blues_r")
    .hide(axis="index")
)
display(tabla_materias)

Código,Materia,Año,Evaluaciones,Promedio,Asistencia %,No aprobación %,Desaprobados %,Ausentismo %
FIS-01,Física,1,827,5.37,70.3,55.1,47.4,7.7
BD-03,Base de Datos III,4,845,5.52,69.8,55.0,47.5,7.6
PI-01,Procesamiento de Imágenes,5,811,5.64,70.8,52.0,44.8,7.3
PL-01,Programación Lógica,1,857,5.69,69.6,51.9,43.3,8.6
ALG-01,Algoritmos,1,843,5.71,70.6,50.7,42.9,7.7
SO-01,Sistemas Operativos,3,817,5.68,69.4,50.6,43.2,7.3


### 3.2 · Comparativo por carrera

In [20]:
sql_carrera = """
SELECT
    c.nombre_carrera,
    COUNT(DISTINCT f.sk_estudiante) AS alumnos_activos,
    ROUND(AVG(f.nota), 2) AS promedio_carrera,
    ROUND(AVG(f.pct_asistencia), 1) AS asistencia_media_pct,
    ROUND(100.0 * SUM(CASE WHEN NOT f.es_aprobado THEN 1 ELSE 0 END) / COUNT(f.id_hecho), 1) AS pct_no_aprobacion
FROM dw_academico.fact_rendimiento_academico f
JOIN dw_academico.dim_carrera c ON f.sk_carrera = c.sk_carrera
GROUP BY c.nombre_carrera
ORDER BY pct_no_aprobacion DESC;
"""
df_carrera = pd.read_sql(text(sql_carrera), pg_engine)

# La paleta se arma con las carreras que existan en el DW (no está hardcodeada)
COLOR_CARRERA.update(
    {nombre: PALETA_CARRERAS[i % len(PALETA_CARRERAS)]
     for i, nombre in enumerate(df_carrera["nombre_carrera"])}
)

display(
    df_carrera.rename(
        columns={
            "nombre_carrera": "Carrera",
            "alumnos_activos": "Alumnos",
            "promedio_carrera": "Promedio",
            "asistencia_media_pct": "Asistencia %",
            "pct_no_aprobacion": "No aprobación %",
        }
    )
    .style.format(
        {"Alumnos": "{:,.0f}", "Promedio": "{:.2f}", "Asistencia %": "{:.1f}", "No aprobación %": "{:.1f}"}
    )
    .background_gradient(subset=["No aprobación %"], cmap="Reds")
    .hide(axis="index")
)


Carrera,Alumnos,Promedio,Asistencia %,No aprobación %
Ingeniería en Sistemas,200,5.61,70.1,52.6


### 3.3 · Patrón: asistencia vs. aprobación

In [21]:
sql_patrones = """
SELECT
    CASE
        WHEN pct_asistencia < 50 THEN '1. Crítica (<50%)'
        WHEN pct_asistencia < 75 THEN '2. Regular (50% - 74%)'
        ELSE '3. Óptima (>=75%)'
    END AS tramo_asistencia,
    COUNT(*) AS evaluaciones,
    ROUND(AVG(nota), 2) AS nota_promedio,
    ROUND(100.0 * SUM(CASE WHEN NOT es_aprobado THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_reprobacion
FROM dw_academico.fact_rendimiento_academico
GROUP BY 1
ORDER BY 1;
"""
df_patrones = pd.read_sql(text(sql_patrones), pg_engine)
display(
    df_patrones.rename(
        columns={
            "tramo_asistencia": "Tramo de asistencia",
            "evaluaciones": "Evaluaciones",
            "nota_promedio": "Nota promedio",
            "pct_reprobacion": "Reprobación %",
        }
    )
    .style.format({"Evaluaciones": "{:,.0f}", "Nota promedio": "{:.2f}", "Reprobación %": "{:.1f}"})
    .background_gradient(subset=["Reprobación %"], cmap="Reds")
    .hide(axis="index")
)

# --- Contraste con la evidencia: ¿existe realmente la relación? ---
corr_asistencia = pd.read_sql(
    text("""
        SELECT
            COALESCE(corr(nota, pct_asistencia), 0) AS r_nota,
            COALESCE(corr(es_aprobado::int, pct_asistencia), 0) AS r_aprobado
        FROM dw_academico.fact_rendimiento_academico
    """),
    pg_engine,
)
r_nota = float(corr_asistencia["r_nota"].iloc[0])
r_aprobado = float(corr_asistencia["r_aprobado"].iloc[0])

print(f"Correlación nota ~ asistencia: {r_nota:+.3f}")
print(f"Correlación aprobación ~ asistencia: {r_aprobado:+.3f}")
if abs(r_nota) < 0.15:
    print(
        "⚠️ No hay relación lineal entre asistencia y nota con estos datos. "
        "No corresponde afirmar que a menor asistencia hay mayor reprobación: "
        "la tabla de tramos lo muestra plana."
    )
else:
    print(
        "✅ La asistencia sí discrimina: la reprobación baja a medida que sube el tramo "
        "de asistencia. La relación es cuantificable y se puede usar en el semáforo."
    )


Tramo de asistencia,Evaluaciones,Nota promedio,Reprobación %
1. Crítica (<50%),690,5.56,52.9
2. Regular (50% - 74%),"2,220",5.64,52.9
3. Óptima (>=75%),"2,090",5.58,52.1


Correlación nota ~ asistencia: -0.004
Correlación aprobación ~ asistencia: -0.000
⚠️ No hay relación lineal entre asistencia y nota con estos datos. No corresponde afirmar que a menor asistencia hay mayor reprobación: la tabla de tramos lo muestra plana.


## 4 · Visualizaciones

> Todos los gráficos usan una paleta consistente y están preparados para presentación.
> Las figuras también se exportan como HTML autónomo al final del notebook.

### 4.1 · Tasa de no aprobación por materia

In [22]:
df1 = df_materias.sort_values("pct_no_aprobacion").copy()

fig1 = go.Figure()
fig1.add_bar(
    y=df1["nombre_materia"],
    x=df1["pct_desaprobados_reales"],
    name="Desaprobados",
    orientation="h",
    marker=dict(color="#DC2626", line=dict(color="white", width=1)),
    text=df1["pct_desaprobados_reales"],
    texttemplate="%{x:.1f}%",
    textposition="inside",
    hovertemplate="<b>%{y}</b><br>Desaprobados: %{x:.1f}%<extra></extra>",
)
fig1.add_bar(
    y=df1["nombre_materia"],
    x=df1["pct_ausentismo"],
    name="Ausentes",
    orientation="h",
    marker=dict(color="#F59E0B", line=dict(color="white", width=1)),
    text=df1["pct_ausentismo"],
    texttemplate="%{x:.1f}%",
    textposition="inside",
    hovertemplate="<b>%{y}</b><br>Ausentes: %{x:.1f}%<extra></extra>",
)
for _, fila in df1.iterrows():
    fig1.add_annotation(
        x=fila["pct_no_aprobacion"],
        y=fila["nombre_materia"],
        text=f"  {fila['pct_no_aprobacion']:.1f}%",
        showarrow=False,
        xanchor="left",
        font=dict(size=12, color="#475569"),
    )
fig1.update_layout(barmode="stack")
estilo_fig(
    fig1,
    "Tasa de no aprobación por materia",
    "Desaprobados + ausentes · ordenado por total",
    height=470,
    grid="x",
)
fig1.update_xaxes(
    range=[0, df1["pct_no_aprobacion"].max() * 1.18], ticksuffix="%", title="Porcentaje de evaluaciones"
)
fig1.update_yaxes(title=None)
fig1.show()

### 4.2 · Evolución del promedio académico

In [23]:
def promedio_ponderado(df, grupo):
    base = df.assign(peso=lambda d: d["promedio_cuatrimestral"] * d["evaluaciones"])
    out = base.groupby(grupo, as_index=False).agg(
        promedio=("peso", "sum"), evaluaciones=("evaluaciones", "sum")
    )
    out["promedio"] = (out["promedio"] / out["evaluaciones"]).round(2)
    return out


df_linea = promedio_ponderado(df_evolucion, ["periodo_inscripcion", "nombre_carrera"])
df_global = promedio_ponderado(df_evolucion, "periodo_inscripcion")

fig2 = px.line(
    df_linea,
    x="periodo_inscripcion",
    y="promedio",
    color="nombre_carrera",
    markers=True,
    color_discrete_map=COLOR_CARRERA,
    labels={
        "periodo_inscripcion": "Período (año-cuatrimestre)",
        "promedio": "Nota promedio",
        "nombre_carrera": "Carrera",
    },
)
fig2.update_traces(
    line=dict(width=3),
    marker=dict(size=9, line=dict(color="white", width=1.5)),
    hovertemplate="<b>%{fullData.name}</b><br>%{x}: %{y:.2f}<extra></extra>",
)
fig2.add_scatter(
    x=df_global["periodo_inscripcion"],
    y=df_global["promedio"],
    mode="lines+markers",
    name="Promedio general",
    line=dict(color="#94A3B8", dash="dash", width=2),
    marker=dict(size=6, color="#94A3B8"),
    hovertemplate="<b>Promedio general</b><br>%{x}: %{y:.2f}<extra></extra>",
)
estilo_fig(
    fig2,
    "Evolución del promedio académico",
    "Promedio ponderado por cantidad de evaluaciones · 2023 a 2026",
    height=470,
)
fig2.update_xaxes(type="category")
lo, hi = df_linea["promedio"].min(), df_linea["promedio"].max()
fig2.update_yaxes(range=[round(lo - 0.35, 1), round(hi + 0.35, 1)], title="Nota promedio")
fig2.show()

### 4.3 · Mapa de calor: desaprobación por carrera y año

In [24]:
tabla_heatmap = df_evolucion.pivot_table(
    index="nombre_carrera",
    columns="anio_cursada",
    values="tasa_desaprobacion",
    aggfunc="mean",
).round(1)
# Años como categorías (no todos los años tienen materias)
tabla_heatmap.columns = [f"{int(anio)}° año" for anio in tabla_heatmap.columns]

fig3 = px.imshow(
    tabla_heatmap,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale=["#FFF5F5", "#FFE4E6", "#FECDD3", "#FDA4AF", "#FB7185"],
    labels=dict(x="Año de cursada", y="Carrera", color="% desaprobación"),
)
estilo_fig(
    fig3,
    "Mapa de calor: desaprobación por carrera y año",
    "Porcentaje promedio de evaluaciones no aprobadas",
    height=400,
)
fig3.update_traces(textfont=dict(size=14, color="#0F172A"))
fig3.update_layout(coloraxis_colorbar=dict(title="% desap.", thickness=14, len=0.8))
fig3.update_xaxes(type="category", title="Año de cursada")
fig3.update_yaxes(title=None)
fig3.show()

### 4.4 · Semáforo de riesgo por cohorte de ingreso

In [25]:
df4 = (
    df_alertas.groupby(["cohorte_ingreso", "semaforo_riesgo"])
    .size()
    .reset_index(name="alumnos")
)
df4["pct"] = df4.groupby("cohorte_ingreso")["alumnos"].transform(lambda s: s / s.sum() * 100)
df4["cohorte_ingreso"] = df4["cohorte_ingreso"].astype(str)
# Solo mostramos la etiqueta en segmentos con espacio suficiente
df4["etiqueta"] = df4["pct"].map(lambda v: f"{v:.0f}%" if v >= 10 else "")

fig4 = px.bar(
    df4,
    x="cohorte_ingreso",
    y="pct",
    color="semaforo_riesgo",
    color_discrete_map=COLOR_RIESGO,
    barmode="stack",
    text="etiqueta",
    category_orders={"semaforo_riesgo": ["Riesgo Alto", "Riesgo Medio", "Riesgo Bajo"]},
    labels={
        "cohorte_ingreso": "Cohorte de ingreso",
        "pct": "% de alumnos",
        "semaforo_riesgo": "Nivel de riesgo",
    },
    custom_data=["alumnos"],
)
fig4.update_traces(
    textposition="inside",
    insidetextanchor="middle",
    hovertemplate="<b>Cohorte %{x}</b><br>%{fullData.name}: %{y:.1f}%<br>Alumnos: %{customdata[0]}<extra></extra>",
)
estilo_fig(
    fig4,
    "Semáforo de riesgo por cohorte de ingreso",
    "Distribución porcentual de alumnos en cada nivel",
    height=450,
)
fig4.update_yaxes(ticksuffix="%", range=[0, 100], title="% de alumnos")
fig4.show()

### 4.5 · Distribución general del riesgo

In [26]:
df5 = (
    df_alertas["semaforo_riesgo"]
    .value_counts()
    .rename_axis("riesgo")
    .reset_index(name="alumnos")
)

fig5 = px.pie(
    df5,
    names="riesgo",
    values="alumnos",
    hole=0.62,
    color="riesgo",
    color_discrete_map=COLOR_RIESGO,
    category_orders={"riesgo": ["Riesgo Alto", "Riesgo Medio", "Riesgo Bajo"]},
)
fig5.update_traces(
    textinfo="percent+label",
    textposition="inside",
    insidetextorientation="horizontal",
    marker=dict(line=dict(color="white", width=2)),
    hovertemplate="<b>%{label}</b><br>%{value} alumnos (%{percent})<extra></extra>",
)
estilo_fig(
    fig5,
    "Distribución general del riesgo",
    f"{alumnos} alumnos clasificados por el semáforo de alerta temprana",
    height=430,
)
fig5.update_layout(legend=dict(orientation="h", y=-0.05, x=0.5, xanchor="center"))
fig5.show()

## 5 · Modelo predictivo de riesgo de abandono

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

# =====================================================================
# MODELO A · réplica del semáforo
# ---------------------------------------------------------------------
# La etiqueta (semaforo_riesgo) se calcula con las mismas variables que se
# usan como features. El modelo NO descubre nada: aprende la regla que ya
# escribimos en SQL. Sirve como control de consistencia, no como predictor.
# =====================================================================
df_ml = df_alertas.copy()
df_ml["promedio_general"] = df_ml["promedio_general"].fillna(0)

features = ["asistencia_promedio", "materias_con_aprobacion", "promedio_general", "cantidad_ausentes"]
X = df_ml[features]
y = (df_ml["semaforo_riesgo"] == "Riesgo Alto").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

modelo = RandomForestClassifier(
    n_estimators=300, max_depth=4, random_state=42, class_weight="balanced"
)
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, 1]

res_a = {
    "accuracy": accuracy_score(y_test, y_pred),
    "auc": roc_auc_score(y_test, y_proba),
    "casos": len(y_test),
}
cm = confusion_matrix(y_test, y_pred)

print(f"MODELO A · exactitud {res_a['accuracy']:.1%} | AUC {res_a['auc']:.3f} | n={res_a['casos']}")
print(classification_report(y_test, y_pred, target_names=["Normal/Medio", "Riesgo Alto"], digits=3))
print("⚠️ Modelo A aprende la regla del semáforo (misma información en features y etiqueta).")

# =====================================================================
# MODELO B · abandono observado
# ---------------------------------------------------------------------
# Etiqueta = el alumno deja de registrarse después del período de corte.
# Es un resultado OBSERVADO, no una regla. Las features se calculan solo
# con información ANTERIOR al corte (validación temporal, sin leakage).
# =====================================================================
periodos = pd.read_sql(text("SELECT DISTINCT periodo_inscripcion FROM dw_academico.dim_periodo"), pg_engine)
periodos = sorted(periodos["periodo_inscripcion"])
CUT = os.getenv("CUT_PERIODO") or periodos[-3]


def clave(p: str) -> int:
    anio, cuatri = p.split("-")
    return int(anio) * 10 + int(cuatri)


CUT_KEY = clave(CUT)

df_hist = pd.read_sql(
    text("""
        SELECT e.id_alumno, e.cohorte_ingreso, p.periodo_inscripcion,
               f.sk_materia, f.nota, f.es_aprobado, f.pct_asistencia, f.estado_aprobacion
        FROM dw_academico.fact_rendimiento_academico f
        JOIN dw_academico.dim_estudiante e USING (sk_estudiante)
        JOIN dw_academico.dim_periodo p USING (sk_periodo)
    """),
    pg_engine,
)
df_hist["clave"] = df_hist["periodo_inscripcion"].map(clave)

base = df_hist[df_hist["clave"] <= CUT_KEY]
activos_despues = set(df_hist.loc[df_hist["clave"] > CUT_KEY, "id_alumno"])

feat_b = base.groupby("id_alumno").agg(
    evaluaciones=("nota", "size"),
    asistencia_promedio=("pct_asistencia", "mean"),
    promedio_general=("nota", "mean"),
    tasa_reprobacion=("es_aprobado", lambda s: 100.0 * (1 - s.mean())),
    cantidad_ausentes=("estado_aprobacion", lambda s: int((s == "Ausente").sum())),
)
feat_b["materias_aprobadas"] = (
    base[base["es_aprobado"]].groupby("id_alumno")["sk_materia"].nunique()
    .reindex(feat_b.index).fillna(0)
)
feat_b["cohorte_ingreso"] = base.groupby("id_alumno")["cohorte_ingreso"].first()

# Solo alumnos con historia suficiente y que ya estaban cursando en el corte
filtro = (feat_b["evaluaciones"] >= 4) & (feat_b["cohorte_ingreso"] <= int(CUT[:4]) - 2)
feat_b = feat_b[filtro]

y_b = (~feat_b.index.isin(activos_despues)).astype(int)

res_b = None
if y_b.nunique() < 2:
    print(
        f"\n⚠️ MODELO B no aplicable a este dataset: con corte {CUT} no hay alumnos que "
        f"abandonen (todos siguen activos). Es lo esperado en el dataset base, donde cada "
        f"alumno aparece en todos los períodos. Usá el dataset v2 para la validación real."
    )
else:
    features_b = [
        "asistencia_promedio", "promedio_general", "tasa_reprobacion",
        "cantidad_ausentes", "materias_aprobadas", "evaluaciones",
    ]
    X_b = feat_b[features_b].fillna(0)
    Xb_train, Xb_test, yb_train, yb_test = train_test_split(
        X_b, y_b, test_size=0.25, random_state=42, stratify=y_b
    )
    modelo_b = RandomForestClassifier(
        n_estimators=300, max_depth=4, random_state=42, class_weight="balanced"
    )
    modelo_b.fit(Xb_train, yb_train)
    yb_pred = modelo_b.predict(Xb_test)
    yb_proba = modelo_b.predict_proba(Xb_test)[:, 1]
    cm_b = confusion_matrix(yb_test, yb_pred)

    res_b = {
        "corte": CUT,
        "alumnos": int(len(y_b)),
        "abandonaron": int(y_b.sum()),
        "accuracy": accuracy_score(yb_test, yb_pred),
        "auc": roc_auc_score(yb_test, yb_proba),
        "casos": len(yb_test),
    }
    print(f"\nMODELO B · abandono observado (corte {CUT})")
    print(f"  alumnos evaluados: {res_b['alumnos']} | abandonaron: {res_b['abandonaron']}")
    print(f"  exactitud {res_b['accuracy']:.1%} | AUC {res_b['auc']:.3f} | n={res_b['casos']}")
    print(classification_report(yb_test, yb_pred, target_names=["Sigue activo", "Abandona"], digits=3))


MODELO A · exactitud 78.0% | AUC 0.895 | n=50
              precision    recall  f1-score   support

Normal/Medio      0.868     0.846     0.857        39
 Riesgo Alto      0.500     0.545     0.522        11

    accuracy                          0.780        50
   macro avg      0.684     0.696     0.689        50
weighted avg      0.787     0.780     0.783        50

⚠️ Modelo A aprende la regla del semáforo (misma información en features y etiqueta).


AttributeError: 'numpy.ndarray' object has no attribute 'nunique'

In [28]:
etiquetas_clase = ["Normal/Medio", "Riesgo Alto"]

fig_cm = px.imshow(
    cm,
    text_auto=True,
    aspect="auto",
    x=etiquetas_clase,
    y=etiquetas_clase,
    color_continuous_scale=["#EFF6FF", "#DBEAFE", "#BFDBFE", "#93C5FD", "#60A5FA"],
    labels=dict(x="Predicción", y="Valor real", color="Casos"),
)
estilo_fig(
    fig_cm,
    "Matriz de confusión · modelo A",
    f"Réplica del semáforo · exactitud {res_a['accuracy']:.1%} · 25% reservado para prueba",
    height=380,
)
fig_cm.update_layout(coloraxis_showscale=False)
fig_cm.update_traces(textfont=dict(size=16, color="#0F172A"))
fig_cm.show()

fig_roc_b = None
if res_b:
    from sklearn.metrics import roc_curve

    fpr, tpr, _ = roc_curve(yb_test, yb_proba)
    fig_roc_b = go.Figure()
    fig_roc_b.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name="Random Forest",
                                   line=dict(color="#2563EB", width=3)))
    fig_roc_b.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Azar",
                                   line=dict(color="#94A3B8", dash="dash")))
    estilo_fig(
        fig_roc_b,
        "Curva ROC · abandono observado",
        f"Corte {res_b['corte']} · AUC {res_b['auc']:.3f} · {res_b['alumnos']} alumnos",
        height=420,
        grid="both",
    )
    fig_roc_b.update_xaxes(title="Tasa de falsos positivos")
    fig_roc_b.update_yaxes(title="Tasa de verdaderos positivos")
    fig_roc_b.show()


In [29]:
etiquetas = {
    "asistencia_promedio": "Asistencia promedio",
    "materias_con_aprobacion": "Materias aprobadas",
    "promedio_general": "Promedio general",
    "cantidad_ausentes": "Ausencias",
}
df_imp = (
    pd.DataFrame(
        {
            "Factor": [etiquetas[f] for f in features],
            "Importancia": (modelo.feature_importances_ * 100).round(1),
        }
    )
    .sort_values("Importancia")
)

fig_imp = px.bar(
    df_imp,
    x="Importancia",
    y="Factor",
    orientation="h",
    text="Importancia",
    color="Importancia",
    color_continuous_scale="Blues",
    labels={"Importancia": "Importancia (%)", "Factor": ""},
)
fig_imp.update_traces(texttemplate="%{x:.1f}%", textposition="outside")
estilo_fig(
    fig_imp,
    "Factores del modelo A (réplica del semáforo)",
    "Importancia relativa de cada variable en el modelo",
    height=390,
    grid="x",
)
fig_imp.update_layout(coloraxis_showscale=False)
fig_imp.update_xaxes(
    range=[0, df_imp["Importancia"].max() * 1.3], ticksuffix="%", title="Importancia (%)"
)
fig_imp.update_yaxes(title=None)
fig_imp.show()

fig_imp_b = None
if res_b:
    etiquetas_b = {
        "asistencia_promedio": "Asistencia promedio",
        "promedio_general": "Promedio general",
        "tasa_reprobacion": "Tasa de reprobación",
        "cantidad_ausentes": "Exámenes ausentes",
        "materias_aprobadas": "Materias aprobadas",
        "evaluaciones": "Evaluaciones rendidas",
    }
    df_imp_b = (
        pd.DataFrame(
            {
                "Factor": [etiquetas_b[f] for f in features_b],
                "Importancia": (modelo_b.feature_importances_ * 100).round(1),
            }
        )
        .sort_values("Importancia")
    )
    fig_imp_b = px.bar(
        df_imp_b,
        x="Importancia",
        y="Factor",
        orientation="h",
        text="Importancia",
        color="Importancia",
        color_continuous_scale="Reds",
        labels={"Importancia": "Importancia (%)", "Factor": ""},
    )
    fig_imp_b.update_traces(texttemplate="%{x:.1f}%", textposition="outside")
    estilo_fig(
        fig_imp_b,
        "Factores del abandono observado",
        "Qué variables, medidas antes del corte, anticipan que el alumno deje de cursar",
        height=420,
        grid="x",
    )
    fig_imp_b.update_layout(coloraxis_showscale=False)
    fig_imp_b.update_xaxes(
        range=[0, df_imp_b["Importancia"].max() * 1.3], ticksuffix="%", title="Importancia (%)"
    )
    fig_imp_b.update_yaxes(title=None)
    fig_imp_b.show()


## 6 · Conclusiones

Las conclusiones se generan a partir de los números que calculó el notebook,
no de texto fijo: si los datos cambian, el texto cambia con ellos.


In [30]:
peor = df_materias.iloc[0]
mejor = df_materias.iloc[-1]
pct_alto = df_alertas["semaforo_riesgo"].eq("Riesgo Alto").mean() * 100
pct_bajo = df_alertas["semaforo_riesgo"].eq("Riesgo Bajo").mean() * 100
tasa_global = float((df_materias["pct_no_aprobacion"] * df_materias["total_evaluaciones"]).sum() / total_eval)

lineas = [
    f"- **No aprobación global: {tasa_global:.1f}%** sobre {total_eval:,} evaluaciones "
    f"de {alumnos} alumnos.",
    f"- La materia más crítica es **{peor['nombre_materia']}** ({peor['pct_no_aprobacion']:.1f}% de no aprobación); "
    f"la menos afectada, **{mejor['nombre_materia']}** ({mejor['pct_no_aprobacion']:.1f}%).",
    f"- El semáforo clasifica **{pct_alto:.1f}%** de los alumnos en Riesgo Alto y "
    f"**{pct_bajo:.1f}%** en Riesgo Bajo.",
]

if abs(r_nota) < 0.15:
    lineas.append(
        f"- **La asistencia no explica la nota en este dataset** (r = {r_nota:+.3f}): "
        "los tramos de asistencia tienen la misma reprobación. El dato sintético se generó "
        "con las columnas independientes, así que la relación no existe y no se afirma."
    )
else:
    lineas.append(
        f"- **La asistencia sí se relaciona con la nota** (r = {r_nota:+.3f}): la reprobación "
        "cae de forma sostenida a medida que sube el tramo de asistencia."
    )

if res_b:
    lineas.append(
        f"- **Abandono observado** (corte {res_b['corte']}): el modelo detecta a los alumnos que "
        f"dejan de cursar con AUC **{res_b['auc']:.3f}** y exactitud **{res_b['accuracy']:.1%}**, "
        f"usando solo información anterior al corte. El factor con más peso es "
        f"**{df_imp_b.iloc[-1]['Factor']}** ({df_imp_b.iloc[-1]['Importancia']:.1f}%)."
    )
else:
    lineas.append(
        f"- **No hay abandono observable** en este dataset (todos los alumnos aparecen hasta el "
        f"último período), por eso el modelo predictivo no se puede validar acá: se valida sobre "
        f"el dataset ampliado (v2), que sí tiene trayectorias que terminan."
    )

lineas.append(
    "- El modelo A **no es un predictor**: reproduce el semáforo con las mismas variables. "
    "Su valor es confirmar qué factores pesan, y así está documentado."
)
lineas.append(
    "- El Data Warehouse permite pasar de datos crudos a decisiones accionables de alerta "
    "temprana con consultas auditables."
)

display(Markdown("\n".join(lineas)))


- **No aprobación global: 52.6%** sobre 5,000 evaluaciones de 200 alumnos.
- La materia más crítica es **Física** (55.1% de no aprobación); la menos afectada, **Sistemas Operativos** (50.6%).
- El semáforo clasifica **21.5%** de los alumnos en Riesgo Alto y **1.5%** en Riesgo Bajo.
- **La asistencia no explica la nota en este dataset** (r = -0.004): los tramos de asistencia tienen la misma reprobación. El dato sintético se generó con las columnas independientes, así que la relación no existe y no se afirma.
- **No hay abandono observable** en este dataset (todos los alumnos aparecen hasta el último período), por eso el modelo predictivo no se puede validar acá: se valida sobre el dataset ampliado (v2), que sí tiene trayectorias que terminan.
- El modelo A **no es un predictor**: reproduce el semáforo con las mismas variables. Su valor es confirmar qué factores pesan, y así está documentado.
- El Data Warehouse permite pasar de datos crudos a decisiones accionables de alerta temprana con consultas auditables.

## 7 · Exportación de gráficos


In [31]:
REPORTS = BASE_DIR / "reports"
REPORTS.mkdir(exist_ok=True)

figuras = {
    "01_no_aprobacion_por_materia": fig1,
    "02_evolucion_promedio": fig2,
    "03_heatmap_desaprobacion": fig3,
    "04_semaforo_por_cohorte": fig4,
    "05_distribucion_riesgo": fig5,
    "06_matriz_confusion": fig_cm,
    "07_importancia_factores": fig_imp,
}
if fig_roc_b is not None:
    figuras["08_roc_abandono"] = fig_roc_b
if fig_imp_b is not None:
    figuras["09_importancia_abandono"] = fig_imp_b

for nombre, fig in figuras.items():
    fig.write_html(REPORTS / f"{nombre}.html", include_plotlyjs="directory")

print(f"✅ {len(figuras)} gráficos exportados a {REPORTS}")


✅ 7 gráficos exportados a /home/gabito/Documentos/Integrador_DB3/reports
